In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch

from src.data.data_manager import create_data_loaders
from src.utils.config_manager import load_config
from src.utils.baselines.feature_extraction import (
    extract_simple_features, extract_bandwise_energy, extract_spectral_features,
    extract_framewise_stats, extract_delta_features, extract_peak_features,
    extract_logmel_features, extract_features
)

In [ ]:
plt.rcParams['figure.figsize'] = (12, 6)

config = load_config()
mode = config['model'].get('type', 'regression')

if mode != 'regression':
    print("WARNING: Naive baselines use regression. Switching to regression mode.")

print(f"Configuration:")
print(f"  - Sample rate: {config['data']['sample_rate']}")
print(f"  - Slice length: {config['data']['slice_length']}s")
print(f"  - Test split: {config['data']['test_split']}")
print(f"  - Val split: {config['data']['val_split']}")

train_loader, val_loader, test_loader, dataset = create_data_loaders(config)

print(f"\nSet sizes:")
print(f"  - Train: {len(train_loader.dataset)} samples")
print(f"  - Val: {len(val_loader.dataset)} samples")
print(f"  - Test: {len(test_loader.dataset)} samples")

In [ ]:
def extract_labels(data_loader):
    labels = []
    for _, label in data_loader:
        if isinstance(label, torch.Tensor):
            labels.extend(label.cpu().numpy())
        else:
            labels.extend(label)
    return np.array(labels)

y_train = extract_labels(train_loader)
y_val = extract_labels(val_loader)
y_test = extract_labels(test_loader)

print("Label shapes:")
print(f"  - Train: {y_train.shape}")
print(f"  - Val: {y_val.shape}")
print(f"  - Test: {y_test.shape}")

y_train = y_train.flatten()
y_val = y_val.flatten()
y_test = y_test.flatten()
print(f"\nStatistics (train):")
print(f"  - Min: {y_train.min():.2f}°C")
print(f"  - Max: {y_train.max():.2f}°C")
print(f"  - Mean: {y_train.mean():.2f}°C")
print(f"  - Median: {np.median(y_train):.2f}°C")
print(f"  - Std: {y_train.std():.2f}°C")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].hist(y_train, bins=30, alpha=0.7, edgecolor='black')
axes[0].set_title('Temperature Distribution - Train')
axes[0].set_xlabel('Temperature [°C]')
axes[0].set_ylabel('Num of Samples')
axes[1].hist(y_val, bins=30, alpha=0.7, edgecolor='black', color='orange')
axes[1].set_title('Temperature Distribution - Val')
axes[1].set_xlabel('Temperature [°C]')
axes[1].set_ylabel('Num of Samples')
axes[2].hist(y_test, bins=30, alpha=0.7, edgecolor='black', color='green')
axes[2].set_title('Temperature Distribution - Test')
axes[2].set_xlabel('Temperature [°C]')
axes[2].set_ylabel('Num of Samples')

plt.tight_layout()
plt.savefig('../../results/baselines/data_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
print("REGRESSION BASELINES")
print("\n" + "="*60)
print("FEATURE EXTRACTION - SIMPLE + ADVANCED")
print("="*60)

# Extract features using the enhanced extractor
X_train, X_val, X_test, y_train_feat, y_val_feat, y_test_feat = extract_features(
    dataset, train_loader, val_loader, test_loader
)

print(f"\nFeature Statistics:")
print(f"  Total features extracted: {X_train.shape[1]}")
print(f"  Train features per sample: {X_train.shape[1]}")
print(f"  Contains:")
print(f"    - Simple spectrogram stats (11)")
print(f"    - Log-mel spectral features (16)")
print(f"    - Global waveform stats (11)")
print(f"    - Spectral handcrafted features (52)")
print(f"    - Pulse/event features (12)")
print(f"    Total: 102 features")

# Handle any NaN or inf values
X_train = np.nan_to_num(X_train, nan=0.0, posinf=0.0, neginf=0.0)
X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
X_test = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)

results = {}
